# This file is concerned with the QA pairs for Blumatix documents
The QA pairs are made up of:
- 1 Reference document
- 20 curated questions
- 20 auto generated answers

The goal of this LLM as a Judge (LLMJ) application is, to test whether the auto generated answer sufficiently answer the question.<br>
For this, multi trace reasoning with majority voting and self-assesment will be used.

## LLM Setup

In [73]:
# --- setup: remove retries, unused vars, fix concurrent name, simpler mkdir ---
from datetime import datetime
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import AsyncOpenAI
import logging
import json

load_dotenv(override=True)

DOCUMENT_STORAGE = Path(os.getenv("DOCUMENT_STORAGE_QA"))
TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

client = AsyncOpenAI(api_key=API_KEY, base_url=ENDPOINT)

CONCURRENT_TASKS = 15

logging_path = Path("../QALogs")
logging_path.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=f"{logging_path}/{DEPLOYMENT_NAME}_{TIMESTAMP}.log",
    filemode="a",
    format="%(asctime)s - %(levelname)s - %(message)s",
    level=logging.INFO,
    force=True,
)
for noisy in ["httpx", "openai", "azure", "urllib3"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)


## Retrieval of documents

In [74]:
from pathlib import Path
import pdfplumber

def load_pdfs_for_context(folder_path: str) -> dict[str, str]:
    """
    Retrieve all .pdf files in a folder, extract text,
    and return a dict mapping filenames to their extracted text.
    """
    folder = Path(folder_path)
    pdf_files = sorted(folder.rglob("*.pdf"))
    context = {}

    for pdf_file in pdf_files:
        try:
            with pdfplumber.open(pdf_file) as pdf:
                text = "\n".join(page.extract_text() or "" for page in pdf.pages)
            context[pdf_file.name] = text.strip()
        except Exception as e:
            print(f"Error reading {pdf_file}: {e}")

    return context

pdf_contents = load_pdfs_for_context(DOCUMENT_STORAGE / "docs")
print(f"Loaded {len(pdf_contents)} PDF documents")

# Load QA pairs from DocumentQA_new.json
qa_file_path = DOCUMENT_STORAGE / "DocumentQA_new.json"
with open(qa_file_path, 'r', encoding='utf-8') as f:
    qa_data = json.load(f)

# Store QA pairs in a dictionary organized by document name
qa_pairs_by_document = {}
for item in qa_data:
    doc_name = item["document_name"]
    qa_pairs_by_document[doc_name] = item["qa_pairs"]

print(f"Loaded QA pairs for {len(qa_pairs_by_document)} documents")
print(f"Total QA pairs: {sum(len(pairs) for pairs in qa_pairs_by_document.values())}")


Loaded 10 PDF documents
Loaded QA pairs for 10 documents
Total QA pairs: 200


## LLM Call

In [75]:
# Updated _timed_request function using client.responses.create API
import time
import asyncio
from openai import BadRequestError

# Semaphore to control concurrent requests
request_semaphore = asyncio.Semaphore(CONCURRENT_TASKS)

CONFIDENCE_CLASSES = [
    "Almost no chance",
    "Highly unlikely",
    "Chances are slight",
    "Unlikely",
    "Less than even",
    "Better than even",
    "Likely",
    "Very good chance",
    "Highly likely",
    "Almost certain"
]

def score_to_verbal(score: float) -> str:
    s = 0.0 if score is None else float(score)
    s = max(0.0, min(1.0, s))
    bounds = [0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
    idx = max(i for i, b in enumerate(bounds) if s >= b)
    return CONFIDENCE_CLASSES[idx]

async def _timed_request(model, instructions, input_text, reasoning=None, max_output_tokens=2000, text=None, user="qa-judge", store=True, timeout_sec=60):
    """Timed request function using client.responses.create API with semaphore control"""
    async with request_semaphore:  # Control concurrent requests
        start = time.time()
        try:
            resp = await asyncio.wait_for(
                client.responses.create(
                    model=model,
                    instructions=instructions,
                    input=input_text,
                    reasoning=reasoning,
                    max_output_tokens=max_output_tokens,
                    text=text,
                    user=user,
                    store=store
                ), 
                timeout=timeout_sec
            )
        
        except (asyncio.TimeoutError, BadRequestError) as e:
            runtime = round(time.time() - start, 2)
            print(f"Request failed with error: {e}")
            return None, {
                "runtime_sec": runtime,
                "input_tokens": None,
                "cached_input_tokens": None,
                "output_tokens": None,
                "total_tokens": None,
                "timed_out": True,
        }

    runtime = round(time.time() - start, 2)
    usage = getattr(resp, "usage", None)
    ptd = getattr(usage, "prompt_tokens_details", None) if usage else None
    cached = getattr(ptd, "cached_tokens", None) if ptd else None
    
    # Logging
    # print(f"Request successful. Runtime: {runtime}s")
    
    return resp, {
        "runtime_sec": runtime,
        "input_tokens": getattr(usage, "input_tokens", None),
        "cached_input_tokens": cached,
        "output_tokens": getattr(usage, "output_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
        "timed_out": False,
    }

def _extract_reasoning_text(response):
    """Extract reasoning text from response - updated for responses.create API"""
    if response and hasattr(response, 'output_text'):
        return response.output_text
    return ""

# Updated QA validation function using the fixed request function
async def run_three_step_qa_validation(question: str, answer: str, pdf_context: str, doc_name: str):
    """
    3-step reasoning process to validate if an answer correctly answers a question given PDF context.
    Returns: True/False/Unknown, reasoning, confidence (0-1)
    """
    
    with open("../QAPrompts/system_prompt.txt", "r") as f:
        system_prompt = f.read()
    
    # Step 1 — CONTEXT ANALYSIS
    resp1, log1 = await _timed_request(
        model=DEPLOYMENT_NAME,
        instructions=system_prompt,
        input_text=(
            "Step 1 — CONTEXT ANALYSIS:\n"
            "Analyze the provided document context to understand what information is available to answer the question. "
            "Identify key facts, data points, and relevant sections. "
            "Do NOT evaluate the answer yet - just understand what the document says.\n\n"
            f"--- DOCUMENT CONTEXT START ---\n{pdf_context}\n--- DOCUMENT CONTEXT END ---\n\n"
            f"--- QUESTION ---\n{question}\n"
        ),
        reasoning={"effort": "low"},
        text={"verbosity": "low"}
    )
    
    if resp1 is None:
        return None
    step1_notes = _extract_reasoning_text(resp1)
    
    # Step 2 — ANSWER EVALUATION  
    resp2, log2 = await _timed_request(
        model=DEPLOYMENT_NAME,
        instructions=system_prompt,
        input_text=(
            "Step 2 — ANSWER EVALUATION:\n"
            "Now evaluate the provided answer against the question and document context. "
            "Check for accuracy, completeness, and whether it properly addresses the question. "
            "Consider if the answer contains incorrect information, missing key points, or irrelevant details. "
            "Do NOT provide your final judgment yet.\n\n"
            f"--- PREVIOUS CONTEXT ANALYSIS ---\n{step1_notes}\n\n"
            f"--- QUESTION ---\n{question}\n\n"
            f"--- ANSWER TO EVALUATE ---\n{answer}\n"
        ),
        reasoning={"effort": "low"},
        text={"verbosity": "low"}
    )
    
    if resp2 is None:
        return None
    step2_notes = _extract_reasoning_text(resp2)
    
    # Step 3 — FINAL JUDGMENT
    resp3, log3 = await _timed_request(
        model=DEPLOYMENT_NAME,
        instructions=system_prompt,
        input_text=(
            "Step 3 — FINAL JUDGMENT:\n"
            "Based on your analysis, provide your final evaluation. Return ONLY a single valid JSON object with this exact structure:\n"
            "{\n"
            '  "judgment": "<True|False|Incomplete|Unknown>",\n'
            '  "reasoning": "<detailed explanation of your decision>",\n'
            '  "confidence": <numeric value between 0.0 and 1.0>\n'
            "}\n\n"
            "Where:\n"
            "- True: The answer correctly and sufficiently answers the question based on the document\n"
            "- False: The answer is incorrect, incomplete, or doesn't properly answer the question\n"
            "- Incomplete: The answer is partially correct, but lacks important information. Always add whether you think the answer is true or false (Incomplete|True or Incomplete|False).\n"
            "- Unknown: Cannot determine due to insufficient information in the document\n"
            "- confidence: How certain you are of your judgment (0.0 = very uncertain, 1.0 = very certain)\n\n"
            f"--- CONTEXT ANALYSIS ---\n{step1_notes}\n\n"
            f"--- ANSWER EVALUATION ---\n{step2_notes}\n"
        ),
        reasoning={"effort": "low"},
        text={"verbosity": "low"}
    )
    
    if resp3 is None:
        return None
    
    try:
        result = json.loads(resp3.output_text)
        
        # Add verbal confidence description
        confidence_verbal = score_to_verbal(result.get("confidence", 0.0))
        
        # Calculate total tokens and cost (simplified)
        total_input_tokens = (log1.get("input_tokens", 0) + log2.get("input_tokens", 0) + log3.get("input_tokens", 0))
        total_output_tokens = (log1.get("output_tokens", 0) + log2.get("output_tokens", 0) + log3.get("output_tokens", 0))
        
        final_result = {
            "document_name": doc_name,
            "question": question,
            "answer": answer,
            "judgment": result.get("judgment"),
            "reasoning": result.get("reasoning"),
            "confidence_numeric": result.get("confidence", 0.0),
            "confidence_verbal": confidence_verbal,
            "step_analysis": {
                "step1_context": step1_notes,
                "step2_evaluation": step2_notes
            },
            "llm_logs": {
                "step1": log1,
                "step2": log2, 
                "step3": log3
            },
            "token_usage": {
                "total_input_tokens": total_input_tokens,
                "total_output_tokens": total_output_tokens
            }
        }
        
        return final_result
        
    except json.JSONDecodeError as e:
        logging.error(f"Failed to parse JSON response: {e}")
        return None

print("QA validation function loaded successfully!")


QA validation function loaded successfully!


In [76]:
# Multithreaded batch processing functions
import asyncio
from typing import List, Dict, Any, Tuple
from datetime import datetime

async def process_qa_pair(qa_pair: Dict[str, Any], pdf_context: str, doc_name: str) -> Tuple[Dict[str, Any], str]:
    """Process a single QA pair and return the result with a unique identifier"""
    qa_id = f"{doc_name}_{qa_pair.get('Question', '')[:50]}..."
    
    try:
        result = await run_three_step_qa_validation(
            question=qa_pair["Question"],
            answer=qa_pair["Detailed Answer"], 
            pdf_context=pdf_context,
            doc_name=doc_name
        )
        return result, qa_id
    except Exception as e:
        logging.error(f"Error processing QA pair {qa_id}: {e}")
        return None, qa_id

async def process_document_qa_pairs(doc_name: str, qa_pairs: List[Dict[str, Any]], 
                                   pdf_context: str, max_questions: int = None) -> List[Dict[str, Any]]:
    """Process all QA pairs for a single document concurrently"""
    
    # Limit number of questions if specified
    if max_questions:
        qa_pairs = qa_pairs[:max_questions]
    
    print(f"Processing {len(qa_pairs)} QA pairs for document: {doc_name}")
    
    # Create tasks for concurrent processing
    tasks = [
        process_qa_pair(qa_pair, pdf_context, doc_name) 
        for qa_pair in qa_pairs
    ]
    
    # Execute all tasks concurrently
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    # Process results and handle exceptions
    processed_results = []
    successful = 0
    failed = 0
    
    for i, (result, qa_id) in enumerate(results):
        if isinstance(result, Exception):
            logging.error(f"Exception in QA pair {qa_id}: {result}")
            failed += 1
        elif result is None:
            print(f"Failed to process QA pair: {qa_id}")
            failed += 1
        else:
            processed_results.append(result)
            successful += 1
    
    print(f"Document {doc_name}: {successful} successful, {failed} failed")
    return processed_results

async def process_all_documents(qa_pairs_by_document: Dict[str, List[Dict[str, Any]]], 
                               pdf_contents: Dict[str, str],
                               max_docs: int = None, 
                               max_questions_per_doc: int = None) -> List[Dict[str, Any]]:
    """Process all documents and their QA pairs with progress tracking"""
    
    # Limit number of documents if specified
    docs_to_process = list(qa_pairs_by_document.keys())
    if max_docs:
        docs_to_process = docs_to_process[:max_docs]
    
    print(f"Starting concurrent processing of {len(docs_to_process)} documents...")
    print(f"Concurrent tasks limit: {CONCURRENT_TASKS}")
    start_time = datetime.now()
    
    # Create tasks for all documents
    document_tasks = []
    for doc_name in docs_to_process:
        if doc_name in pdf_contents:
            task = process_document_qa_pairs(
                doc_name, 
                qa_pairs_by_document[doc_name], 
                pdf_contents[doc_name], 
                max_questions_per_doc
            )
            document_tasks.append(task)
        else:
            print(f"Warning: No PDF content found for document {doc_name}")
    
    # Process all documents concurrently
    all_results = await asyncio.gather(*document_tasks, return_exceptions=True)
    
    # Flatten results and handle exceptions
    final_results = []
    total_successful = 0
    total_failed = 0
    
    for i, doc_results in enumerate(all_results):
        if isinstance(doc_results, Exception):
            logging.error(f"Exception processing document {docs_to_process[i]}: {doc_results}")
            total_failed += 1
        else:
            final_results.extend(doc_results)
            total_successful += len(doc_results)
    
    end_time = datetime.now()
    processing_time = (end_time - start_time).total_seconds()
    
    print(f"\nProcessing completed in {processing_time:.2f} seconds")
    print(f"Total results: {len(final_results)}")
    print(f"Success rate: {total_successful}/{total_successful + total_failed}")
    
    return final_results

print("Multithreaded processing functions loaded successfully!")


Multithreaded processing functions loaded successfully!


In [77]:
# Multithreaded processing example - much faster than sequential processing!

# Configuration for testing
NUM_DOCS = 1
NUM_QUESTIONS = 5

# Run the multithreaded processing
results = await process_all_documents(
    qa_pairs_by_document=qa_pairs_by_document,
    pdf_contents=pdf_contents,
    max_docs=NUM_DOCS,
    max_questions_per_doc=NUM_QUESTIONS
)

# Display sample results
print(f"\n=== SAMPLE RESULTS ({len(results)} total) ===")
for i, result in enumerate(results[:3]):  # Show first 3 results
    if result:
        print(f"\n--- Result {i+1} ---")
        print(f"Document: {result['document_name']}")
        print(f"Question: {result['question'][:100]}...")
        print(f"Judgment: {result['judgment']}")
        print(f"Confidence: {result['confidence_numeric']:.2f} ({result['confidence_verbal']})")
        print(f"Reasoning: {result['reasoning'][:200]}...")
    else:
        print(f"--- Result {i+1}: Failed ---")
    

Starting concurrent processing of 1 documents...
Concurrent tasks limit: 15
Processing 5 QA pairs for document: Accounts Payable Processing_BLU DELTA_Nintex Platform-V2_Nov 2022_de-NBCHRIS.pdf
Document Accounts Payable Processing_BLU DELTA_Nintex Platform-V2_Nov 2022_de-NBCHRIS.pdf: 5 successful, 0 failed

Processing completed in 18.03 seconds
Total results: 5
Success rate: 5/5

=== SAMPLE RESULTS (5 total) ===

--- Result 1 ---
Document: Accounts Payable Processing_BLU DELTA_Nintex Platform-V2_Nov 2022_de-NBCHRIS.pdf
Question: Wie verbessert die Integration von BLU DELTA mit der Nintex Prozessplattform den Rechnungseingangspr...
Judgment: Incomplete|True
Confidence: 0.86 (Highly likely)
Reasoning: The answer largely reflects the document: it correctly describes automation of manual invoice steps (printing, assignment, approval), one-click transfer to financial accounting, use of AI (NLP/deep le...

--- Result 2 ---
Document: Accounts Payable Processing_BLU DELTA_Nintex Platform-V2_Nov

In [79]:
# Save results to file for analysis
import json
from pathlib import Path

def save_results_to_file(results: List[Dict[str, Any]], filename: str = None):
    """Save processing results to a JSON file"""
    if filename is None:
        filename = f"qa_results_{DEPLOYMENT_NAME}_{TIMESTAMP}.json"
    
    output_dir = Path("../QAOutput")
    output_dir.mkdir(exist_ok=True)
    
    output_path = output_dir / filename
    
    # Prepare results for JSON serialization
    serializable_results = []
    for result in results:
        if result:
            # Create a clean copy without non-serializable objects
            clean_result = {
                "document_name": result.get("document_name"),
                "question": result.get("question"),
                "answer": result.get("answer"),
                "judgment": result.get("judgment"),
                "reasoning": result.get("reasoning"),
                "confidence_numeric": result.get("confidence_numeric"),
                "confidence_verbal": result.get("confidence_verbal"),
                "token_usage": result.get("token_usage", {}),
                "timestamp": TIMESTAMP
            }
            serializable_results.append(clean_result)
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump({
            "metadata": {
                "timestamp": TIMESTAMP,
                "model": DEPLOYMENT_NAME,
                "total_results": len(serializable_results),
                "concurrent_tasks": CONCURRENT_TASKS
            },
            "results": serializable_results
        }, f, indent=2, ensure_ascii=False)
    
    print(f"Results saved to: {output_path}")
    return output_path

# Example usage (uncomment to save results):
if 'results' in locals() and results:
    save_results_to_file(results)

print("Results saving function loaded successfully!")


Results saved to: ..\QAOutput\qa_results_gpt-5-mini_2025-09-24_13-34-25.json
Results saving function loaded successfully!
